In [ ]:
import pandas as pd
import seaborn as sns

In [ ]:
df = pd.read_csv("kernel-breakdown.csv", sep=";")
df["kernel"] = df["kernel"].map(
    {
        "flashinfer:sampling_from_logits": "FlashInfer",
        "fused-matmul-sample": "FMS",
    }
)
df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

sns.set_theme(style="whitegrid")
sns.set_context("talk")

# Aggregate (important if there could be duplicates)
df_agg = df.groupby(["batch_size", "kernel", "step"], as_index=False).agg(
    time_ms=("time_ms", "sum")
)

# Get unique batch sizes
batch_sizes = sorted(df_agg["batch_size"].unique())

fig, axes = plt.subplots(1, len(batch_sizes), figsize=(5 * len(batch_sizes), 4), sharey=True)

if len(batch_sizes) == 1:
    axes = [axes]

palette = sns.color_palette("tab10")

bar_width = 0.6

for ax, bs in zip(axes, batch_sizes):
    df_bs = df_agg[df_agg["batch_size"] == bs]

    # Pivot to wide format for stacking
    pivot = df_bs.pivot(index="kernel", columns="step", values="time_ms").fillna(0)

    # x locations for bars
    x = range(len(pivot.index))

    bottom = None
    for i, step in enumerate(pivot.columns):
        bar = ax.bar(x, pivot[step], bottom=bottom, label=step, color=palette[i], width=bar_width)
        # Show the duration and step name on each bar segment
        for rect, v in zip(bar.patches, pivot[step]):
            height = rect.get_height()
            if height > 0:
                # Combine step name and duration
                display_text = f"{step.capitalize()}\n{v:.3f} ms"
                ax.text(
                    rect.get_x() + rect.get_width() / 2,
                    rect.get_y() + height / 2,
                    display_text,
                    ha="center",
                    va="center",
                    fontsize=10,
                    color="black",
                )
        bottom = pivot[step] if bottom is None else bottom + pivot[step]

    ax.set_title(f"Batch size = {bs}")
    ax.set_xlabel("")
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.grid(axis="y", alpha=0.5)
    ax.tick_params(axis="x")

axes[0].set_ylabel("Time (ms)")
# axes[-1].legend(title="Step", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("kernel-breakdown.png", dpi=300)
plt.show()